# CF3 — A8 Scaling · Hierarchical Interfaces

- Canon (anchor-only; do not duplicate): [../../Complete-Formalisms/CF3_A8_Scaling_Hierarchical_Interfaces.md](../../Complete-Formalisms/CF3_A8_Scaling_Hierarchical_Interfaces.md)
- Purpose: exhaustive, testable code recreation of CF3 formal statements — implements blocking/rescaling and computes the collapse envelope E_max. Thresholds live in KPIs; canon owns definitions (no duplication). Artifacts must route via io_paths in production; this is not a full experiment.

Navigation anchors (canon registries):
- RG / blocking operator and scaling map: [../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-136](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-136)
- GENERIC / scale program (dimensionless groups): see A6 notes in AXIOMS and linked equations.


## Step 1 — Build a synthetic profile and block at multiple scales

We create a 1D synthetic field (Gaussian with a small oscillatory component) and apply simple average-blocking at factors $s\in\{2,4,8\}$.

Blocking is a test surrogate for coarse-graining; the exact operator is defined in canon. Here, we demonstrate numerically that curves can be compared across scales after rescaling to dimensionless axes.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def make_field(N=1024):
    x = np.linspace(-4.0, 4.0, N)
    y = np.exp(-x**2) * (1.0 + 0.1*np.cos(3.5*x))
    return x, y

def block_avg(y, s: int):
    n = (len(y)//s)*s
    yb = y[:n].reshape(-1, s).mean(axis=1)
    return yb

x, y = make_field(1024)
y2 = block_avg(y, 2)
y4 = block_avg(y, 4)
y8 = block_avg(y, 8)
print({'y_len': len(y), 'y2_len': len(y2), 'y4_len': len(y4), 'y8_len': len(y8)})


## Step 2 — Dimensionless rescaling and collapse envelope $E_{\max}$

For each curve, rescale abscissa to $t\in[0,1]$ and normalize amplitude to unit max. Then define a simple collapse envelope against a chosen reference (the finest resolution curve):

$$
E_{\max}(\{z_k\})\;=\;\max_{t\in[0,1]}\;\max_k\,\big|z_\mathrm{ref}(t)-\operatorname{interp}(z_k; t)\big|.
$$

A small $E_{\max}$ indicates good collapse after blocking/rescaling. Thresholds for acceptance live in canon KPIs (this notebook only demonstrates testability).

In [ ]:
def rescale_to_unit(y):
    t = np.linspace(0.0, 1.0, len(y))
    denom = max(1e-15, np.max(np.abs(y)))
    z = y/denom
    return t, z

t1, z1 = rescale_to_unit(y)
t2, z2 = rescale_to_unit(y2)
t4, z4 = rescale_to_unit(y4)
t8, z8 = rescale_to_unit(y8)

def envelope_max(ref_t, ref_z, curves):
    em = 0.0
    for (tk, zk) in curves:
        zk_interp = np.interp(ref_t, tk, zk)
        em = max(em, float(np.max(np.abs(ref_z - zk_interp))))
    return em

E_max = envelope_max(t1, z1, [(t2,z2),(t4,z4),(t8,z8)])
print({'E_max': E_max})


## Step 3 — Sensitivity sweep (optional): oscillation strength

We vary the synthetic oscillation amplitude $\alpha\in\{0.0,\,0.05,\,0.1,\,0.2\}$ and report $E_{\max}$ to illustrate that collapse quality is measurable. Larger oscillations generally make collapse harder (larger $E_{\max}$).

In [ ]:
def make_field_alpha(N=1024, alpha=0.1):
    x = np.linspace(-4.0, 4.0, N)
    y = np.exp(-x**2) * (1.0 + alpha*np.cos(3.5*x))
    return x, y

summary = []
for alpha in [0.0, 0.05, 0.1, 0.2]:
    x, y = make_field_alpha(1024, alpha)
    y2, y4, y8 = block_avg(y,2), block_avg(y,4), block_avg(y,8)
    t1,z1 = rescale_to_unit(y)
    t2,z2 = rescale_to_unit(y2)
    t4,z4 = rescale_to_unit(y4)
    t8,z8 = rescale_to_unit(y8)
    E = envelope_max(t1, z1, [(t2,z2),(t4,z4),(t8,z8)])
    summary.append({'alpha': alpha, 'E_max': E})

for row in summary:
    print(row)


## Summary — Minimal, falsifiable checks

- After blocking/rescaling, we compute a collapse envelope $E_{\max}$ quantifying agreement across scales.
- A parameter sweep (oscillation strength) moves $E_{\max}$ in a sensible direction, demonstrating measurable sensitivity.

This confirms the CF3 formalism is runnable and testable; thresholds (e.g., acceptable envelope magnitude) are governed by canon KPIs and are out of scope for this minimal notebook.

## Repro notes

- Determinism: pure NumPy; IEEE-754 double precision assumed; no RNG in core steps.
- No files are written here; production gates/figures must route via repository IO policy.